# Sentiment Consistency Pipeline — Local Run

Local counterpart of `run_pipeline_colab.ipynb`. No Drive mount, no `git
clone` (you're already in the repo), no upload step — this assumes:

1. The `venv/` at the project root already has all dependencies installed
   (`pip install -r requirements.txt`), including `ipykernel`.
2. Set this notebook's kernel to that venv before running anything
   (VS Code: top-right kernel picker → select the `venv` interpreter).
3. `cresci-2017.csv/datasets_full.csv/` (the dataset zips) is already on
   disk at the project root.

**Do not run the pipeline cell below while another `python -m src.pipeline`
run is already going** (e.g. one started from a terminal) — running two at
once just makes both slower fighting over the same CPU.

## 1. Move to the project root

Jupyter sets the working directory to wherever this notebook file lives
(`notebooks/`); everything else in this project expects to run from the
repo root, so fix that up front.

In [ ]:
import os
import pathlib

if pathlib.Path.cwd().name == "notebooks":
    os.chdir(pathlib.Path.cwd().parent)

PROJECT_ROOT = pathlib.Path.cwd()
print("Working directory:", PROJECT_ROOT)

## 2. Extract Cresci-2017 into data/raw/

Skip this cell if `data/raw/` is already populated (e.g. you already ran
the pipeline once from a terminal) — it's safe to re-run either way, just
redundant.

Uses genuine_accounts + all social_spambots + all traditional_spambots,
matching the proposal's "3 bot types" composition. To also test
`fake_followers.csv.zip` as an extra class, add it to `SUBSETS`.

In [ ]:
import zipfile

LOCAL_DATA_DIR = PROJECT_ROOT / "cresci-2017.csv" / "datasets_full.csv"
RAW_DIR = PROJECT_ROOT / "data" / "raw"
RAW_DIR.mkdir(parents=True, exist_ok=True)

SUBSETS = [
    "genuine_accounts.csv.zip",
    "social_spambots_1.csv.zip",
    "social_spambots_2.csv.zip",
    "social_spambots_3.csv.zip",
    "traditional_spambots_1.csv.zip",
    "traditional_spambots_2.csv.zip",
    "traditional_spambots_3.csv.zip",
    "traditional_spambots_4.csv.zip",
]

for name in SUBSETS:
    zip_path = LOCAL_DATA_DIR / name
    print("Extracting", zip_path)
    with zipfile.ZipFile(zip_path) as zf:
        zf.extractall(RAW_DIR)

print(sorted(p.name for p in RAW_DIR.iterdir()))

## 3. Run the pipeline

This is the slow step — expect roughly 30-45 minutes on this machine (see
the progress bars for live status). Runs via `sys.executable` so it's
guaranteed to use this notebook's own kernel/venv, not whatever `python`
happens to resolve to on PATH.

In [ ]:
import subprocess
import sys

subprocess.run([sys.executable, "-m", "src.pipeline"], cwd=PROJECT_ROOT, check=True)

## 4. Generate the interpretation report

In [ ]:
subprocess.run([sys.executable, "-m", "src.report"], cwd=PROJECT_ROOT, check=True)

## 5. Inspect the raw result tables

In [ ]:
import pandas as pd

summary = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "experiment_summary.csv")
summary

In [ ]:
mcnemar = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "mcnemar_significance.csv")
mcnemar

In [ ]:
importance = pd.read_csv(PROJECT_ROOT / "results" / "tables" / "feature_importance.csv")
importance.head(10)

## 6. View the interpretation report and plots

In [ ]:
from IPython.display import Markdown, display

report_path = PROJECT_ROOT / "results" / "tables" / "interpretation_report.md"
display(Markdown(report_path.read_text(encoding="utf-8")))

In [ ]:
from IPython.display import Image, display

figures_dir = PROJECT_ROOT / "results" / "figures"
display(Image(str(figures_dir / "f1_comparison.png")))
display(Image(str(figures_dir / "feature_importance.png")))